In [ ]:
from pyspark.sql.functions import col

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_products_table", "olist_products")
dbutils.widgets.text("raw_category_translation_table", "product_category_name_translation")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("products_table", "products_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_products_table_name = dbutils.widgets.get("raw_olist_products_table")
raw_category_translation_table_name = dbutils.widgets.get("raw_category_translation_table")

silver_schema = dbutils.widgets.get("silver_schema")
products_table_name = dbutils.widgets.get("products_table")

In [ ]:
raw_olist_products_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_products_table_name}")
raw_category_translation_df = spark.table(f"{catalog}.{bronze_schema}.{raw_category_translation_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{products_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{products_table_name} (
            productId STRING,
            productCategoryName STRING,
            productNameLength INT,
            productDescriptionLength INT,
            productPhotosQty INT,
            productWeightG INT,
            productLengthCm INT,
            productHeightCm INT,
            productWidthCm INT
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
products_silver_df = (
    raw_olist_products_df.alias("products")
    .join(
        raw_category_translation_df.alias("category"),
        on="product_category_name",
        how="inner"
    )
    .where(col("products.product_id").rlike("^[0-9a-fA-F]{32}$"))
    .select(
        col("products.product_id").alias("productId"),
        col("category.product_category_name").alias("productCategoryName"),
        col("products.product_name_length").alias("productNameLength"),
        col("products.product_description_length").alias("productDescriptionLength"),
        col("products.product_photos_qty").alias("productPhotosQty"),
        col("products.product_weight_g").alias("productWeightG"),
        col("products.product_length_cm").alias("productLengthCm"),
        col("products.product_height_cm").alias("productHeightCm"),
        col("products.product_width_cm").alias("productWidthCm")
    )
)

In [ ]:
products_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.{products_table_name}")